# Notebook v5 — ablasi ukuran sampel dan sensitivitas parameter SMOTE

Dua eksperimen yang diminta Reviewer 2 secara eksplisit:

> *Perform ablation tests on various training sample sizes — for example, 29, 50, 100, and 500
> samples, depending on data availability — to evaluate the consistency of SMOTE's performance
> improvement.*
>
> *Include a sensitivity analysis of SMOTE parameters, particularly the number of nearest neighbors
> and the oversampling ratio.*

**Bagian T — ablasi ukuran.** Eksperimen terkontrol diulang pada n = 29, 50, 100 dan 500 sampel
latih per kategori. Subsampelnya dibuat **bersarang**: satu permutasi acak per (seed, fold,
kategori), lalu n pertama diambil. Jadi subset n = 29 benar-benar termuat di dalam n = 50, yang
termuat di n = 100, dan seterusnya. Dengan begitu perbedaan antar ukuran adalah efek ukuran, bukan
efek kebetulan sampel mana yang terpilih.

Karena bersarang, titik n = 29 di sini **tidak identik** dengan Tabel 2 dan 4 di naskah, yang
memakai `choice` alih-alih permutasi. Perbedaannya justru berguna: ia menjadi pemeriksaan ulang
seberapa besar hasil n = 29 bergantung pada subsampel mana yang kebetulan terambil.

**Bagian V — sensitivitas parameter.** Pada titik kontrol n = 29, jumlah tetangga terdekat
divariasikan (k = 3, 5, 7, 9) dengan rasio tetap, lalu rasio oversampling divariasikan
(0,25 / 0,5 / 1,0 kali target) dengan k tetap di 5.

**Catatan penting soal Infiltration.** Kategori ini hanya punya 36 record, sehingga lipatan
latihnya berisi 28 atau 29 sampel. Ia otomatis dilewati pada ukuran 50, 100 dan 500, dan sel
analisis melaporkan pelewatan itu apa adanya, bukan menyembunyikannya.

| Bagian | Isi | Perkiraan waktu |
|---|---|---|
| 0–1 | Setup dan baca cache data bersih | < 1 menit |
| T | Ablasi, satu ukuran per sesi | ±2–3 jam per ukuran |
| U | Analisis ablasi | < 1 menit |
| V | Sensitivitas parameter SMOTE | ±3 jam |
| W | Analisis sensitivitas | < 1 menit |

Notebook ini **membaca cache** `clean_data.npz` yang dibuat notebook v3, jadi tidak mengunduh
CIC-IDS2017 ulang. Jalankan v3 bagian 0–3 lebih dulu bila cache itu belum ada.


In [ ]:
# ====== PENGATURAN ======
N_FOLDS      = 5
TARGET       = 24000          # target penyeimbangan, identik dengan naskah
RF_TREES     = 100
XGB_TREES    = 300
XGB_DEPTH    = 6
XGB_LR       = 0.1
RANDOM_STATE = 42

# --- Bagian T: ablasi ukuran sampel ---
ABL_SIZES    = [29, 50, 100, 500]
CTRL_N       = 29             # titik kontrol naskah, dipakai sebagai pembanding
ABL_SEEDS    = [42]           # tambahkan 43 bila waktu mengizinkan
ABL_MODELS   = ["rf", "xgb"]  # MLP dilewati: kaki terlemah di naskah
ABL_METHODS  = ["base", "smote"]
RUN_SIZES    = [29]           # SATU ukuran per sesi; lalu [50], [100], [500]
ABL_MIN_FRAC = 0.9            # jalankan dengan sampel seadanya bila >= 90% dari ukuran diminta;
                              # di bawah itu kategori dilewati. Infiltration punya 28 sampai 29
                              # sampel per lipatan, jadi n = 29 tetap berjalan di kelima lipatan
                              # dengan jumlah sebenarnya dicatat di kolom n_train.

# --- Bagian V: sensitivitas parameter SMOTE ---
SENS_N       = 29             # dijalankan pada titik kontrol
SENS_K       = [3, 5, 7, 9]   # rasio tetap di TARGET
SENS_RATIO   = [0.25, 0.5, 1.0]   # k tetap di 5; 1,0 = TARGET
SENS_MODEL   = "rf"
SENS_SEEDS   = [42]

TIME_BUDGET_MIN = 170         # berhenti rapi sebelum Colab memutus
RUN_T = True
RUN_V = False                 # nyalakan setelah bagian T selesai

import os, time, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTDIR = "/content/drive/MyDrive/juti_rerun"
except Exception:
    OUTDIR = "./juti_rerun"
os.makedirs(OUTDIR, exist_ok=True)
print("Folder hasil:", OUTDIR)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "scikit-learn==1.6.1", "imbalanced-learn==0.14.2", "xgboost"], check=False)
import sklearn, imblearn, xgboost
print("scikit-learn", sklearn.__version__, "| imbalanced-learn", imblearn.__version__,
      "| xgboost", xgboost.__version__)

## 1. Baca data bersih dari cache

Notebook ini memakai berkas `clean_data.npz` yang ditulis notebook v3. Kalau berkas itu belum ada,
jalankan dulu notebook v3 bagian 0 sampai 3 — proses unduh dan pembersihannya tidak diulang di sini
supaya tidak ada dua jalur preprocessing yang bisa menyimpang satu sama lain.


In [ ]:
CLEAN = os.path.join(OUTDIR, "clean_data.npz")
if not os.path.exists(CLEAN):
    raise FileNotFoundError(
        "clean_data.npz tidak ditemukan di " + OUTDIR + "\n"
        "Jalankan notebook v3 (JUTI_rerun_v3.ipynb) bagian 0 sampai 3 lebih dulu; "
        "sel itu mengunduh, membersihkan, lalu menyimpan cache yang dibaca di sini.")
z = np.load(CLEAN, allow_pickle=True)
X_all = z["X_all"]
y_all = z["y_all"].astype(str)
FEATURES = list(z["features"])
CLASSES = sorted(np.unique(y_all))
TARGET_CATS = ["Botnet", "PortScan", "Web Attack", "Infiltration", "DoS", "DDoS"]

print(f"{X_all.shape[0]:,} record x {X_all.shape[1]} fitur")
print(pd.Series(y_all).value_counts().to_string())

# berapa sampel latih yang tersedia per kategori pada tiap fold
from sklearn.model_selection import StratifiedKFold
print("\nSampel latih tersedia per kategori (seed 42):")
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
avail = {c: [] for c in TARGET_CATS}
for tr, _ in skf.split(X_all, y_all):
    for c in TARGET_CATS:
        avail[c].append(int((y_all[tr] == c).sum()))
for c in TARGET_CATS:
    lo, hi = min(avail[c]), max(avail[c])
    usable = [n for n in ABL_SIZES if n <= lo]
    print(f"  {c:13s} {lo:6,}–{hi:6,}  ->  ukuran yang bisa dijalankan: {usable}")

## T. Ablasi ukuran sampel

Desainnya sama persis dengan eksperimen terkontrol di naskah, kecuali satu hal: jumlah sampel latih
kategori sasaran tidak lagi dipaku di 29, melainkan divariasikan.

Subsampelnya bersarang. Untuk setiap (seed, fold, kategori) dibuat satu permutasi acak dari seluruh
indeks kategori itu, lalu `n` pertama diambil. Karena permutasinya sama untuk semua `n`, subset yang
lebih kecil selalu termuat di dalam yang lebih besar.

Jalankan **satu ukuran per sesi** lewat `RUN_SIZES`. Kategori yang sampelnya tidak cukup untuk
ukuran yang diminta dicatat sebagai dilewati, lengkap dengan jumlah yang tersedia.


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

CLS2ID = {c: i for i, c in enumerate(CLASSES)}
ID2CLS = np.array(CLASSES)

def make_model(name, seed):
    if name == "rf":
        return RandomForestClassifier(n_estimators=RF_TREES, n_jobs=-1, random_state=seed)
    if name == "xgb":
        return XGBClassifier(n_estimators=XGB_TREES, max_depth=XGB_DEPTH, learning_rate=XGB_LR,
                             subsample=0.9, colsample_bytree=0.9, tree_method="hist",
                             objective="multi:softmax", num_class=len(CLASSES),
                             n_jobs=-1, random_state=seed, verbosity=0)
    raise ValueError(name)

def fit_predict(name, seed, Xs, ys, Xte):
    m = make_model(name, seed)
    if name == "xgb":
        m.fit(Xs, np.array([CLS2ID[v] for v in ys], dtype=int))
        return ID2CLS[np.asarray(m.predict(Xte)).astype(int)]
    m.fit(Xs, ys)
    return np.asarray(m.predict(Xte)).astype(str)

def smote_strategy(y, target=None):
    target = TARGET if target is None else target
    vc = pd.Series(y).value_counts()
    return {c: target for c, n in vc.items() if n < target}

def counts_for(y_true, y_pred, c):
    return (int(((y_pred == c) & (y_true == c)).sum()),
            int(((y_pred == c) & (y_true != c)).sum()),
            int(((y_pred != c) & (y_true == c)).sum()))

CKT = os.path.join(OUTDIR, "secT_ablation.csv")
COLS_T = ["seed", "fold", "category", "size", "model", "method",
          "n_train", "tp", "fp", "fn", "skipped", "available", "fit_seconds"]
doneT = pd.read_csv(CKT) if os.path.exists(CKT) else pd.DataFrame(columns=COLS_T)
keysT = set(zip(doneT.seed, doneT.fold, doneT.category, doneT["size"],
                doneT.model, doneT.method)) if len(doneT) else set()

SLICE = [(s, f, c, n, mo, me) for s in ABL_SEEDS for f in range(N_FOLDS)
         for c in TARGET_CATS for n in RUN_SIZES
         for mo in ABL_MODELS for me in ABL_METHODS]
todo = [k for k in SLICE if k not in keysT]
print(f"Ukuran sesi ini: {RUN_SIZES} | perlu dilatih: {len(todo)} dari {len(SLICE)}")
if TIME_BUDGET_MIN:
    print(f"Batas waktu: {TIME_BUDGET_MIN} menit")

if not RUN_T:
    print("\n>>> RUN_T = False, jadi tidak ada model yang dilatih di sel ini.")
    print(">>> Ubah RUN_T = True di sel Pengaturan, jalankan ulang sel itu, lalu sel ini.")

t_start, n_run, stopped = time.time(), 0, False
for seed in (ABL_SEEDS if RUN_T else []):
    if stopped: break
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (tr, te) in enumerate(skf.split(X_all, y_all)):
        if stopped: break
        if not any((seed, fold, c, n, mo, me) not in keysT
                   for c in TARGET_CATS for n in RUN_SIZES
                   for mo in ABL_MODELS for me in ABL_METHODS):
            continue
        sc = StandardScaler().fit(X_all[tr])
        Xtr_f, Xte_f = sc.transform(X_all[tr]), sc.transform(X_all[te])
        ytr_f, yte_f = y_all[tr], y_all[te]
        for c in TARGET_CATS:
            if stopped: break
            idx_all = np.where(ytr_f == c)[0]
            # satu permutasi per (seed, fold, kategori) -> subsampel bersarang
            perm = np.random.default_rng(seed * 100 + fold).permutation(idx_all)
            for n in RUN_SIZES:
                if stopped: break
                n_eff = min(n, len(idx_all))
                if n_eff < ABL_MIN_FRAC * n:
                    for mo in ABL_MODELS:
                        for me in ABL_METHODS:
                            if (seed, fold, c, n, mo, me) in keysT:
                                continue
                            doneT = pd.concat([doneT, pd.DataFrame([dict(
                                seed=seed, fold=fold, category=c, size=n, model=mo, method=me,
                                n_train=np.nan, tp=np.nan, fp=np.nan, fn=np.nan,
                                skipped=1, available=len(idx_all), fit_seconds=0.0)])],
                                ignore_index=True)
                            keysT.add((seed, fold, c, n, mo, me))
                    doneT.to_csv(CKT, index=False)
                    print(f"seed {seed} fold {fold} {c:12s} n={n:<4} DILEWATI "
                          f"(hanya {len(idx_all)} sampel tersedia)")
                    continue
                idx_c = perm[:n_eff]
                if n_eff < n:
                    print(f"seed {seed} fold {fold} {c:12s} n={n:<4} DIPOTONG ke {n_eff} "
                          f"(hanya {len(idx_all)} sampel tersedia)")
                keep_idx = np.r_[np.where(ytr_f != c)[0], idx_c]
                Xk, yk = Xtr_f[keep_idx], ytr_f[keep_idx]
                for me in ABL_METHODS:
                    if stopped: break
                    todo_m = [mo for mo in ABL_MODELS
                              if (seed, fold, c, n, mo, me) not in keysT]
                    if not todo_m:
                        continue
                    Xs, ys = ((SMOTE(sampling_strategy=smote_strategy(yk), k_neighbors=5,
                                     random_state=seed).fit_resample(Xk, yk))
                              if me == "smote" else (Xk, yk))
                    for mo in todo_m:
                        if TIME_BUDGET_MIN and (time.time()-t_start)/60 >= TIME_BUDGET_MIN:
                            stopped = True; break
                        t0 = time.time()
                        pred = fit_predict(mo, seed, Xs, ys, Xte_f)
                        dt = time.time() - t0
                        tp, fp, fn = counts_for(yte_f, pred, c)
                        doneT = pd.concat([doneT, pd.DataFrame([dict(
                            seed=seed, fold=fold, category=c, size=n, model=mo, method=me,
                            n_train=len(idx_c), tp=tp, fp=fp, fn=fn,
                            skipped=0, available=len(idx_all),
                            fit_seconds=round(dt, 1))])], ignore_index=True)
                        doneT.to_csv(CKT, index=False)
                        keysT.add((seed, fold, c, n, mo, me)); n_run += 1
                        left = len([k for k in SLICE if k not in keysT])
                        avg = (time.time()-t_start)/max(n_run, 1)
                        print(f"seed {seed} fold {fold} {c:12s} n={n:<4} {mo:3s} {me:5s} "
                              f"{dt:5.0f} s | sisa {left} ≈ {left*avg/60:.0f} menit")

left = len([k for k in SLICE if k not in keysT])
if stopped:
    print(f"\nBerhenti karena batas waktu. Sisa {left} model. Jalankan ulang sel ini "
          f"dengan RUN_SIZES yang sama untuk melanjutkan.")
elif left == 0:
    print(f"\nUkuran {RUN_SIZES} selesai. Ganti RUN_SIZES ke ukuran berikutnya, "
          f"atau jalankan bagian U bila semua ukuran sudah ada.")

## U. Analisis ablasi

Tiga pertanyaan yang dijawab sel ini, dan ketiganya langsung menjawab Reviewer 2:

1. **Apakah manfaat SMOTE menyusut seiring bertambahnya sampel?** Itu yang diperkirakan kalau
   manfaatnya memang menutup defisit deteksi.
2. **Apakah urutan kategori bertahan?** Korelasi Spearman antara peringkat ΔF1 pada n = 29 dan pada
   setiap ukuran lebih besar. Kalau urutannya berubah, klaim naskah bahwa identitas kategori yang
   menentukan hanya berlaku pada satu titik kontrol.
3. **Seberapa besar hasil n = 29 bergantung pada subsampel yang kebetulan terambil?** Titik n = 29
   di sini memakai permutasi, naskah memakai `choice`, jadi selisih keduanya adalah ukuran
   langsung dari derau subsampel.


In [ ]:
from scipy.stats import spearmanr

f1 = lambda tp, fp, fn: 200.0*tp/max(2*tp+fp+fn, 1)
d = pd.read_csv(CKT)
ok = d[d.skipped == 0].copy()
sk = d[d.skipped == 1]
if len(sk):
    s = sk.groupby(["category", "size"])["available"].first().reset_index()
    print("Dilewati karena sampel tidak cukup:")
    print(s.to_string(index=False), "\n")
short = ok.groupby(["category", "size"])["n_train"].mean().reset_index()
short = short[short.n_train < short["size"] - 1e-9]
if len(short):
    print("Dipotong: jumlah sampel latih sebenarnya di bawah ukuran nominal")
    print(short.rename(columns={"size": "nominal", "n_train": "rata-rata sebenarnya"})
          .round(2).to_string(index=False), "\n")

rows = []
for (c, n, mo), sub in ok.groupby(["category", "size", "model"]):
    b = sub[sub.method == "base"].set_index(["seed", "fold"]).sort_index()
    s_ = sub[sub.method == "smote"].set_index(["seed", "fold"]).sort_index()
    k = b.index.intersection(s_.index)
    if not len(k):
        continue
    b, s_ = b.loc[k], s_.loc[k]
    f1b = np.array([f1(*r) for r in b[["tp", "fp", "fn"]].values])
    f1s = np.array([f1(*r) for r in s_[["tp", "fp", "fn"]].values])
    rows.append(dict(category=c, size=int(n), model=mo, n_pairs=len(k),
                     base_F1=f1b.mean(), smote_F1=f1s.mean(), delta=(f1s-f1b).mean(),
                     delta_sd=(f1s-f1b).std(ddof=0),
                     R_c=100.0*(b.fn.sum()-s_.fn.sum())/max(b.fn.sum(), 1)))
A = pd.DataFrame(rows)
if not len(A):
    raise SystemExit(
        "Belum ada hasil ablasi di " + CKT + ".\n"
        "Jalankan bagian T lebih dulu dengan RUN_T = True dan satu ukuran di RUN_SIZES.")

print("=== 1. Manfaat SMOTE (ΔF1) menurut ukuran sampel ===")
for mo in sorted(A.model.unique()):
    piv = A[A.model == mo].pivot_table(index="category", columns="size", values="delta")
    piv = piv.reindex(index=[c for c in TARGET_CATS if c in piv.index])
    print(f"\n-- {mo.upper()} --")
    print(piv.round(1).to_string())
    base = A[A.model == mo].pivot_table(index="category", columns="size", values="base_F1")
    print(f"   (baseline F1 pada ukuran terkecil dan terbesar)")
    print(base.reindex(index=piv.index).round(1).iloc[:, [0, -1]].to_string())

print("\n\n=== 2. Apakah urutan kategori bertahan? ===")
for mo in sorted(A.model.unique()):
    sub = A[A.model == mo]
    sizes = sorted(sub["size"].unique())
    ref = sub[sub["size"] == sizes[0]].set_index("category")["delta"]
    print(f"\n-- {mo.upper()}, pembanding n = {sizes[0]} --")
    for n in sizes[1:]:
        cur = sub[sub["size"] == n].set_index("category")["delta"]
        common = ref.index.intersection(cur.index)
        if len(common) < 4:
            print(f"  n = {n:<4}: hanya {len(common)} kategori, terlalu sedikit")
            continue
        r, p = spearmanr(ref.loc[common], cur.loc[common])
        print(f"  n = {n:<4}: rho = {r:+.2f} (p = {p:.3f}), {len(common)} kategori"
              + ("  *" if p < 0.05 else ""))

print(f"\n\n=== 3. Derau subsampel pada n = {CTRL_N} ===")
print(f"Membandingkan titik n = {CTRL_N} di sini (permutasi) dengan Tabel 4 naskah (choice).")
CKG = os.path.join(OUTDIR, "secG_controlled_v3.csv")
if os.path.exists(CKG) and CTRL_N in set(A["size"]):
    g = pd.read_csv(CKG)
    g = g[g.method.isin(["base", "smote"]) & g.model.isin(ABL_MODELS) & (g.seed.isin(ABL_SEEDS))]
    rows2 = []
    for (c, mo), sub in g.groupby(["category", "model"]):
        b = sub[sub.method == "base"].set_index(["seed", "fold"]).sort_index()
        s_ = sub[sub.method == "smote"].set_index(["seed", "fold"]).sort_index()
        k = b.index.intersection(s_.index)
        if not len(k): continue
        b, s_ = b.loc[k], s_.loc[k]
        f1b = np.array([f1(*r) for r in b[["tp", "fp", "fn"]].values])
        f1s = np.array([f1(*r) for r in s_[["tp", "fp", "fn"]].values])
        rows2.append(dict(category=c, model=mo, delta_naskah=(f1s-f1b).mean(),
                          base_naskah=f1b.mean()))
    G = pd.DataFrame(rows2)
    cmp = A[A["size"] == CTRL_N].merge(G, on=["category", "model"], how="inner")
    cmp["selisih_delta"] = cmp.delta - cmp.delta_naskah
    cmp["selisih_base"] = cmp.base_F1 - cmp.base_naskah
    print(cmp[["category", "model", "base_F1", "base_naskah", "selisih_base",
               "delta", "delta_naskah", "selisih_delta"]].round(2).to_string(index=False))
    print(f"\nSelisih |Δ| rata-rata: {cmp.selisih_delta.abs().mean():.2f} poin, "
          f"maksimum {cmp.selisih_delta.abs().max():.2f} poin")
else:
    print(f"  (butuh secG_controlled_v3.csv dan hasil ablasi n = {CTRL_N})")

A.to_csv(os.path.join(OUTDIR, "v5_ablation_summary.csv"), index=False)
print("\nRingkasan disimpan: v5_ablation_summary.csv")

## V. Sensitivitas parameter SMOTE

Dijalankan pada titik kontrol n = 29 dengan Random Forest sebagai keluarga acuan.

Dua sumbu, diubah satu per satu agar pengaruhnya bisa dipisahkan:

* **Jumlah tetangga terdekat** k = 3, 5, 7, 9 dengan rasio tetap di target penuh. Nilai k = 5 adalah
  yang dipakai naskah.
* **Rasio oversampling** 0,25 / 0,5 / 1,0 kali target 24.000, dengan k tetap di 5. Nilai 1,0 adalah
  yang dipakai naskah.

Yang ingin diketahui bukan konfigurasi mana yang terbaik, melainkan apakah kesimpulan naskah
bergeser bila parameternya digeser. Kalau peringkat kategori bertahan di seluruh pengaturan, pilihan
k = 5 dan rasio penuh bukan penentu hasil.


In [ ]:
CKV = os.path.join(OUTDIR, "secV_smote_sensitivity.csv")
COLS_V = ["seed", "fold", "category", "k", "ratio", "model", "tp", "fp", "fn", "fit_seconds"]
doneV = pd.read_csv(CKV) if os.path.exists(CKV) else pd.DataFrame(columns=COLS_V)
keysV = set(zip(doneV.seed, doneV.fold, doneV.category, doneV.k, doneV.ratio)) if len(doneV) else set()

# konfigurasi: ubah k dengan rasio penuh, lalu ubah rasio dengan k = 5
CONFIGS = [(k, 1.0) for k in SENS_K] + [(5, r) for r in SENS_RATIO if r != 1.0]
CONFIGS = sorted(set(CONFIGS))
print("Konfigurasi yang diuji (k, rasio):", CONFIGS)
TOTAL = len(SENS_SEEDS) * N_FOLDS * len(TARGET_CATS) * len(CONFIGS)
print(f"Total {TOTAL} model | sudah ada {len(keysV)}")

if not RUN_V:
    print("\n>>> RUN_V = False, jadi tidak ada model yang dilatih di sel ini.")
    print(">>> Ubah RUN_V = True di sel Pengaturan, jalankan ulang sel itu, lalu sel ini.")
    print(">>> Selama RUN_V masih False, bagian W tidak punya data untuk dibaca.")

t_start, n_run, stopped = time.time(), 0, False
for seed in (SENS_SEEDS if RUN_V else []):
    if stopped: break
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (tr, te) in enumerate(skf.split(X_all, y_all)):
        if stopped: break
        if not any((seed, fold, c, k, r) not in keysV
                   for c in TARGET_CATS for k, r in CONFIGS):
            continue
        sc = StandardScaler().fit(X_all[tr])
        Xtr_f, Xte_f = sc.transform(X_all[tr]), sc.transform(X_all[te])
        ytr_f, yte_f = y_all[tr], y_all[te]
        for c in TARGET_CATS:
            if stopped: break
            idx_all = np.where(ytr_f == c)[0]
            perm = np.random.default_rng(seed * 100 + fold).permutation(idx_all)
            idx_c = perm[:min(SENS_N, len(idx_all))]
            keep_idx = np.r_[np.where(ytr_f != c)[0], idx_c]
            Xk, yk = Xtr_f[keep_idx], ytr_f[keep_idx]
            for k, ratio in CONFIGS:
                if (seed, fold, c, k, ratio) in keysV:
                    continue
                if TIME_BUDGET_MIN and (time.time()-t_start)/60 >= TIME_BUDGET_MIN:
                    stopped = True; break
                kk = min(k, len(idx_c) - 1)
                t0 = time.time()
                Xs, ys = SMOTE(sampling_strategy=smote_strategy(yk, int(TARGET*ratio)),
                               k_neighbors=kk, random_state=seed).fit_resample(Xk, yk)
                pred = fit_predict(SENS_MODEL, seed, Xs, ys, Xte_f)
                dt = time.time() - t0
                tp, fp, fn = counts_for(yte_f, pred, c)
                doneV = pd.concat([doneV, pd.DataFrame([dict(
                    seed=seed, fold=fold, category=c, k=k, ratio=ratio, model=SENS_MODEL,
                    tp=tp, fp=fp, fn=fn, fit_seconds=round(dt, 1))])], ignore_index=True)
                doneV.to_csv(CKV, index=False); keysV.add((seed, fold, c, k, ratio)); n_run += 1
                left = TOTAL - len(keysV)
                avg = (time.time()-t_start)/max(n_run, 1)
                print(f"seed {seed} fold {fold} {c:12s} k={k} r={ratio:<4} {dt:5.0f} s "
                      f"| sisa {left} ≈ {left*avg/60:.0f} menit")
if stopped:
    print("\nBerhenti karena batas waktu. Jalankan ulang sel ini untuk melanjutkan.")
elif RUN_V:
    print("\nBagian V selesai.")

## W. Analisis sensitivitas

Keluarannya dibandingkan terhadap baseline tanpa resampling yang sudah ada, sehingga ΔF1 di sini
setara dengan yang dilaporkan naskah. Baris terakhir memeriksa apakah peringkat kategori bergeser
antar pengaturan — itulah yang menentukan apakah pilihan k = 5 dan rasio penuh menyetir kesimpulan.


In [ ]:
from scipy.stats import spearmanr
f1 = lambda tp, fp, fn: 200.0*tp/max(2*tp+fp+fn, 1)
V = pd.read_csv(CKV) if os.path.exists(CKV) else pd.DataFrame()
if not len(V):
    raise SystemExit(
        "Bagian V belum menghasilkan apa pun.\n"
        "Berkas " + CKV + " tidak ada atau kosong.\n"
        "Penyebab paling sering: RUN_V masih False di sel Pengaturan, sehingga sel V\n"
        "selesai dalam nol detik tanpa melatih satu model pun. Ubah RUN_V = True,\n"
        "jalankan ulang sel Pengaturan, lalu jalankan sel V sampai selesai.")
V["f1"] = [f1(*r) for r in V[["tp", "fp", "fn"]].values]

# baseline tanpa resampling: dari ablasi n = 29 kalau ada, kalau tidak dari secG
base = None
if os.path.exists(CKT):
    t = pd.read_csv(CKT)
    t = t[(t.skipped == 0) & (t["size"] == SENS_N) & (t.model == SENS_MODEL) & (t.method == "base")]
    if len(t):
        t["f1"] = [f1(*r) for r in t[["tp", "fp", "fn"]].values]
        base = t.groupby(["seed", "fold", "category"])["f1"].mean()
if base is None:
    raise SystemExit(
        "Tidak ada baseline n = " + str(SENS_N) + " untuk model " + SENS_MODEL.upper() +
        " di secT_ablation.csv.\n"
        "Bagian W membandingkan hasil sensitivitas terhadap baseline tanpa resampling\n"
        "pada titik kontrol, jadi bagian T harus sudah dijalankan dengan\n"
        "RUN_SIZES = [" + str(SENS_N) + "] dan RUN_T = True.")

V = V.join(base.rename("f1_base"), on=["seed", "fold", "category"])
V["delta"] = V.f1 - V.f1_base
piv = V.pivot_table(index="category", columns=["k", "ratio"], values="delta")
piv = piv.reindex(index=[c for c in TARGET_CATS if c in piv.index])
print("=== ΔF1 menurut parameter SMOTE (model " + SENS_MODEL.upper() + ", n = "
      + str(SENS_N) + ") ===\n")
print(piv.round(1).to_string())

print("\n=== Pengaruh jumlah tetangga, rasio tetap penuh ===")
sub = V[V.ratio == 1.0].pivot_table(index="category", columns="k", values="delta")
sub = sub.reindex(index=[c for c in TARGET_CATS if c in sub.index])
print(sub.round(1).to_string())
ref = sub[5] if 5 in sub.columns else sub.iloc[:, 0]
for kk in sub.columns:
    if kk == 5: continue
    r, p = spearmanr(ref, sub[kk])
    print(f"  urutan k=5 vs k={kk}: rho = {r:+.2f} (p = {p:.3f})")

print("\n=== Pengaruh rasio oversampling, k tetap 5 ===")
sub2 = V[V.k == 5].pivot_table(index="category", columns="ratio", values="delta")
sub2 = sub2.reindex(index=[c for c in TARGET_CATS if c in sub2.index])
print(sub2.round(1).to_string())
ref2 = sub2[1.0] if 1.0 in sub2.columns else sub2.iloc[:, -1]
for rr in sub2.columns:
    if rr == 1.0: continue
    r, p = spearmanr(ref2, sub2[rr])
    print(f"  urutan rasio=1,0 vs rasio={rr}: rho = {r:+.2f} (p = {p:.3f})")

V.to_csv(os.path.join(OUTDIR, "v5_sensitivity_detail.csv"), index=False)
piv.to_csv(os.path.join(OUTDIR, "v5_sensitivity_summary.csv"))
print("\nDisimpan: v5_sensitivity_summary.csv dan v5_sensitivity_detail.csv")